In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType
from decimal import Decimal
from pyspark.sql.functions import to_date

columns_trans= StructType([
    StructField("AccountId", IntegerType(), True),
    StructField("TranDate", StringType(), True), 
    StructField("TranAmt", DecimalType(10, 2), True)
])

data = [
    (1, '2011-01-01', Decimal('500.00')),
    (1, '2011-01-15', Decimal('50.00')),
    (1, '2011-01-22', Decimal('250.00')),
    (1, '2011-01-24', Decimal('75.00')),
    (1, '2011-01-26', Decimal('125.00')),
    (1, '2011-01-28', Decimal('175.00')),
    (2, '2011-01-01', Decimal('500.00')),
    (2, '2011-01-15', Decimal('50.00')),
    (2, '2011-01-22', Decimal('25.00')),
    (2, '2011-01-23', Decimal('125.00')),
    (2, '2011-01-26', Decimal('200.00')),
    (2, '2011-01-29', Decimal('250.00')),
    (3, '2011-01-01', Decimal('500.00')),
    (3, '2011-01-15', Decimal('50.00')),
    (3, '2011-01-22', Decimal('5000.00')),
    (3, '2011-01-25', Decimal('550.00')),
    (3, '2011-01-27', Decimal('95.00')),
    (3, '2011-01-30', Decimal('2500.00'))
]

transactions = spark.createDataFrame(data, columns_trans)
transactions = transactions.withColumn("TranDate", to_date(transactions["TranDate"], "yyyy-MM-dd"))

columns_log = StructType([
    StructField("RowID", IntegerType(), True),
    StructField("FName", StringType(), True),
    StructField("Salary", IntegerType(), True)
])

data2 = [
    (1, 'George', 800),
    (2, 'Sam', 950),
    (3, 'Diane', 1100),
    (4, 'Nicholas', 1250),
    (5, 'Samuel', 1250),
    (6, 'Patricia', 1300),
    (7, 'Brian', 1500),
    (8, 'Thomas', 1600),
    (9, 'Fran', 2450),
    (10, 'Debbie', 2850),
    (11, 'Mark', 2975),
    (12, 'James', 3000),
    (13, 'Cynthia', 3000),
    (14, 'Christopher', 5000)
]

logical = spark.createDataFrame(data2, columns_log)

transactions.show()
logical.show()

+---------+----------+-------+
|AccountId|  TranDate|TranAmt|
+---------+----------+-------+
|        1|2011-01-01| 500.00|
|        1|2011-01-15|  50.00|
|        1|2011-01-22| 250.00|
|        1|2011-01-24|  75.00|
|        1|2011-01-26| 125.00|
|        1|2011-01-28| 175.00|
|        2|2011-01-01| 500.00|
|        2|2011-01-15|  50.00|
|        2|2011-01-22|  25.00|
|        2|2011-01-23| 125.00|
|        2|2011-01-26| 200.00|
|        2|2011-01-29| 250.00|
|        3|2011-01-01| 500.00|
|        3|2011-01-15|  50.00|
|        3|2011-01-22|5000.00|
|        3|2011-01-25| 550.00|
|        3|2011-01-27|  95.00|
|        3|2011-01-30|2500.00|
+---------+----------+-------+

+-----+-----------+------+
|RowID|      FName|Salary|
+-----+-----------+------+
|    1|     George|   800|
|    2|        Sam|   950|
|    3|      Diane|  1100|
|    4|   Nicholas|  1250|
|    5|     Samuel|  1250|
|    6|   Patricia|  1300|
|    7|      Brian|  1500|
|    8|     Thomas|  1600|
|    9|       Fran| 

In [0]:
#SUM
#funkcje okienkowe pozwalaja nam dzielić dane na grupy i działać na konkretnej grupie 
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

#musimy stwoezyc okno podzielone na gruy wedlug accountid i posortowane data 
window = Window.partitionBy("AccountId").orderBy("TranDate")

#policzenie sumy tranamt w konkretnej grupie
suma = transactions.withColumn("suma", sum("TranAmt").over(window))
suma.show()


+---------+----------+-------+-------+
|AccountId|  TranDate|TranAmt|   suma|
+---------+----------+-------+-------+
|        1|2011-01-01| 500.00| 500.00|
|        1|2011-01-15|  50.00| 550.00|
|        1|2011-01-22| 250.00| 800.00|
|        1|2011-01-24|  75.00| 875.00|
|        1|2011-01-26| 125.00|1000.00|
|        1|2011-01-28| 175.00|1175.00|
|        2|2011-01-01| 500.00| 500.00|
|        2|2011-01-15|  50.00| 550.00|
|        2|2011-01-22|  25.00| 575.00|
|        2|2011-01-23| 125.00| 700.00|
|        2|2011-01-26| 200.00| 900.00|
|        2|2011-01-29| 250.00|1150.00|
|        3|2011-01-01| 500.00| 500.00|
|        3|2011-01-15|  50.00| 550.00|
|        3|2011-01-22|5000.00|5550.00|
|        3|2011-01-25| 550.00|6100.00|
|        3|2011-01-27|  95.00|6195.00|
|        3|2011-01-30|2500.00|8695.00|
+---------+----------+-------+-------+



In [0]:
from pyspark.sql.functions import avg, count, min, max, sum,lead,lag,row_number,first,last

#lead-pobiera wartosc z nastepnego rekordu w grupie 
#lag-pobiera wartosc z poprzedniego rekordu w grupie 
#row_number-numeruje wiersze w grupie
transactions= transactions.withColumn("avg", avg("TranAmt").over(window)) \
    .withColumn("sum", sum("TranAmt").over(window)) \
    .withColumn("min", min("TranAmt").over(window)) \
    .withColumn("max", max("TranAmt").over(window)) \
    .withColumn("lead", lead("TranAmt", 1).over(window)) \
    .withColumn("lag", lag("TranAmt", 1).over(window)) \
    .withColumn("row_numb", row_number().over(window))\
    .withColumn("first", first("TranAmt").over(window))\
    .withColumn("last", last("TranAmt").over(window))

transactions.show()

+---------+----------+-------+-----------+-----+------+-------+-------+-------+-------+--------+------+-------+
|AccountId|  TranDate|TranAmt|        avg|count|   min|    max|    sum|   lead|    lag|row_numb| first|   last|
+---------+----------+-------+-----------+-----+------+-------+-------+-------+-------+--------+------+-------+
|        1|2011-01-01| 500.00| 500.000000|    1|500.00| 500.00| 500.00|  50.00|   null|       1|500.00| 500.00|
|        1|2011-01-15|  50.00| 275.000000|    2| 50.00| 500.00| 550.00| 250.00| 500.00|       2|500.00|  50.00|
|        1|2011-01-22| 250.00| 266.666667|    3| 50.00| 500.00| 800.00|  75.00|  50.00|       3|500.00| 250.00|
|        1|2011-01-24|  75.00| 218.750000|    4| 50.00| 500.00| 875.00| 125.00| 250.00|       4|500.00|  75.00|
|        1|2011-01-26| 125.00| 200.000000|    5| 50.00| 500.00|1000.00| 175.00|  75.00|       5|500.00| 125.00|
|        1|2011-01-28| 175.00| 195.833333|    6| 50.00| 500.00|1175.00|   null| 125.00|       6|500.00| 

In [0]:
#mozemy okreslic zakres wierszy jakie beda brane pod uwage w grupie
window2 = Window.partitionBy("AccountId").orderBy("TranDate").rowsBetween(-2, 0)

transactions = transactions .withColumn("avg2", avg("TranAmt").over(window2)) \
    .withColumn("count2", count("TranAmt").over(window2)) \
    .withColumn("min2", min("TranAmt").over(window2)) \
    .withColumn("max2", max("TranAmt").over(window2)) \
    .withColumn("sum2", sum("TranAmt").over(window2)) \
    .withColumn("row_num2", row_number().over(window))

transactions.show()


+---------+----------+-------+-----------+-----+------+-------+-------+-------+-------+--------+------+-------+-----------+------+------+-------+-------+--------+
|AccountId|  TranDate|TranAmt|        avg|count|   min|    max|    sum|   lead|    lag|row_numb| first|   last|       avg2|count2|  min2|   max2|   sum2|row_num2|
+---------+----------+-------+-----------+-----+------+-------+-------+-------+-------+--------+------+-------+-----------+------+------+-------+-------+--------+
|        1|2011-01-01| 500.00| 500.000000|    1|500.00| 500.00| 500.00|  50.00|   null|       1|500.00| 500.00| 500.000000|     1|500.00| 500.00| 500.00|       1|
|        1|2011-01-15|  50.00| 275.000000|    2| 50.00| 500.00| 550.00| 250.00| 500.00|       2|500.00|  50.00| 275.000000|     2| 50.00| 500.00| 550.00|       2|
|        1|2011-01-22| 250.00| 266.666667|    3| 50.00| 500.00| 800.00|  75.00|  50.00|       3|500.00| 250.00| 266.666667|     3| 50.00| 500.00| 800.00|       3|
|        1|2011-01-24|

In [0]:
window_rows = Window.orderBy("Salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
window_range = Window.orderBy("Salary").rangeBetween(Window.unboundedPreceding, Window.currentRow)

# Dodanie kolumn z funkcjami okienkowymi
logical= logical \
    .withColumn("SumByRows", sum("Salary").over(window_rows)) \
    .withColumn("SumByRange", sum("Salary").over(window_range))

# Wyświetlenie wyników
logical.orderBy("RowID").show()

+-----+-----------+------+---------+----------+
|RowID|      FName|Salary|SumByRows|SumByRange|
+-----+-----------+------+---------+----------+
|    1|     George|   800|      800|       800|
|    2|        Sam|   950|     1750|      1750|
|    3|      Diane|  1100|     2850|      2850|
|    4|   Nicholas|  1250|     4100|      5350|
|    5|     Samuel|  1250|     5350|      5350|
|    6|   Patricia|  1300|     6650|      6650|
|    7|      Brian|  1500|     8150|      8150|
|    8|     Thomas|  1600|     9750|      9750|
|    9|       Fran|  2450|    12200|     12200|
|   10|     Debbie|  2850|    15050|     15050|
|   11|       Mark|  2975|    18025|     18025|
|   12|      James|  3000|    21025|     24025|
|   13|    Cynthia|  3000|    24025|     24025|
|   14|Christopher|  5000|    29025|     29025|
+-----+-----------+------+---------+----------+

